<a href="https://colab.research.google.com/github/rick-ssa/clearbank-analise/blob/main/desafio_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


#Importação de bibliotecas



In [34]:
import csv
import json
from datetime import datetime

#Definindo Constantes

In [35]:
# Constantes globais
LIMITE_SUSPEITO = 10000.00
ARQUIVO_CSV = "transacoes.csv"
ARQUIVO_JSON = "relatorio.json"

#Definindo função para validar transação

In [36]:
def validar_transacao(linha):
    """
    Valida uma única linha de transação.
    Retorna o dicionário limpo e convertido se válido, ou None se inválido.
    Usa try/except para conversão de valor e data.
    """
    # 1. Validação de ID (não vazio e numérico)
    if not linha.get("id", "").strip().isdigit():
        return None

    # 2. Validação de cliente_id (não vazio)
    if not linha.get("cliente_id", "").strip():
        return None

    # 3. Validação de tipo (credito ou debito)
    tipo = linha.get("tipo", "").strip().lower()
    if tipo not in ["credito", "debito"]:
        return None

    # 4. Validação de valor (numérico e > 0) com try/except
    try:
        valor = float(linha.get("valor", 0))
        if valor <= 0:
            return None
    except ValueError:
        return None

    # 5. Validação de data (formato AAAA-MM-DD) com try/except
    data_str = linha.get("data", "").strip()
    try:
        data_obj = datetime.strptime(data_str, "%Y-%m-%d")
    except ValueError:
        return None

    # Retorna o registro limpo e estruturado
    return {
        "id": int(linha["id"]),
        "data_obj": data_obj,
        "data_str": data_str,
        "mes_ano": data_obj.strftime("%Y-%m"),
        "cliente_id": linha["cliente_id"].strip(),
        "tipo": tipo,
        "valor": valor,
        "descricao": linha.get("descricao", "").strip(),
        "categoria": linha.get("categoria", "").strip()
    }

#Definindo função para ler transações

In [37]:
def ler_transacoes(nome_arquivo):
    """
    Lê o arquivo CSV utilizando o módulo nativo csv.DictReader.
    Trata o erro de arquivo não encontrado (FileNotFoundError).
    """
    transacoes_validas = []
    total_linhas = 0
    total_invalidas = 0

    try:
        with open(nome_arquivo, mode="r", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)
            for linha in leitor:
                total_linhas += 1
                registro_limpo = validar_transacao(linha)

                if registro_limpo:
                    transacoes_validas.append(registro_limpo)
                else:
                    total_invalidas += 1

    except FileNotFoundError:
        print(f"Erro: O arquivo '{nome_arquivo}' não foi encontrado.")
        print("Certifique-se de criar o arquivo CSV antes de rodar a análise.")

    print(f"Total de linhas lidas: {total_linhas}")
    print(f"Linhas válidas: {len(transacoes_validas)}")
    print(f"Linhas inválidas: {total_invalidas}")

    return transacoes_validas

#Definindo função para gerar relatório

In [38]:
def gerar_relatorio(transacoes):
    """
    Agrupa os dados por mês, calcula as métricas financeiras
    e identifica transações suspeitas.
    """
    resumo_mensal = {}
    transacoes_suspeitas = []

    if not transacoes:
        return {"resumo_mensal": {}, "transacoes_suspeitas": [], "periodo": ("", "")}

    # Ordena as transações por data para obter o período
    transacoes_ordenadas = sorted(transacoes, key=lambda x: x["data_obj"])
    data_mais_antiga = transacoes_ordenadas[0]["data_str"]
    data_mais_recente = transacoes_ordenadas[-1]["data_str"]

    # Cálculo de dias entre a mais antiga e a mais recente
    dias_decorridos = (transacoes_ordenadas[-1]["data_obj"] - transacoes_ordenadas[0]["data_obj"]).days

    for t in transacoes:
        mes = t["mes_ano"]

        # Verifica transações suspeitas
        if t["valor"] > LIMITE_SUSPEITO:
            transacoes_suspeitas.append({
                "id": t["id"],
                "cliente_id": t["cliente_id"],
                "data": t["data_str"],
                "valor": t["valor"]
            })

        # Inicializa o mês no dicionário se não existir
        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "transacoes_lista": []
            }

        resumo_mensal[mes]["quantidade"] += 1
        resumo_mensal[mes]["transacoes_lista"].append(t)

        if t["tipo"] == "credito":
            resumo_mensal[mes]["total_credito"] += t["valor"]
        else:
            resumo_mensal[mes]["total_debito"] += t["valor"]

    # Calcula métricas finais por mês (saldo, média, maior e menor valor)
    metricas_finais = {}
    for mes, dados in sorted(resumo_mensal.items()):
        qtd = dados["quantidade"]
        cred = dados["total_credito"]
        deb = dados["total_debito"]
        saldo = cred - deb
        media = (cred + deb) / qtd if qtd > 0 else 0.0

        lista_valores = [item["valor"] for item in dados["transacoes_lista"]]
        maior_valor = max(lista_valores) if lista_valores else 0.0
        menor_valor = min(lista_valores) if lista_valores else 0.0

        metricas_finais[mes] = {
            "quantidade": qtd,
            "total_credito": round(cred, 2),
            "total_debito": round(deb, 2),
            "saldo": round(saldo, 2),
            "media": round(media, 2),
            "maior_valor": round(maior_valor, 2),
            "menor_valor": round(menor_valor, 2)
        }

    return {
        "periodo": (data_mais_antiga, data_mais_recente),
        "dias_decorridos": dias_decorridos,
        "resumo_mensal": metricas_finais,
        "transacoes_suspeitas": transacoes_suspeitas
    }

#Definindo função para formatar valor para Reais

In [39]:
def formatar_brl(valor):
    """Formata valores numéricos para o padrão monetário brasileiro (R$)."""
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

#Definindo função para exibir relatório

In [40]:
def exibir_relatorio(resultado):
    """Formata e imprime os resultados de forma legível no terminal."""
    print("\n" + "=" * 30 + " RELATÓRIO MENSAL " + "=" * 30)
    periodo = resultado.get("periodo", ("", ""))
    print(f"Período Analisado: {periodo[0]} até {periodo[1]} ({resultado.get('dias_decorridos', 0)} dias entre a primeira e última transação)")

    for mes, metricas in resultado.get("resumo_mensal", {}).items():
        print(f"\nMês: {mes}")
        print(f"  Transações: {metricas['quantidade']}")
        print(f"  Total crédito: {formatar_brl(metricas['total_credito'])}")
        print(f"  Total débito:  {formatar_brl(metricas['total_debito'])}")
        print(f"  Saldo:         {formatar_brl(metricas['saldo'])}")
        print(f"  Média:         {formatar_brl(metricas['media'])}")
        print(f"  Maior valor:   {formatar_brl(metricas['maior_valor'])}")
        print(f"  Menor valor:   {formatar_brl(metricas['menor_valor'])}")

    print("\n" + "=" * 28 + " TRANSAÇÕES SUSPEITAS " + "=" * 28)
    suspeitas = resultado.get("transacoes_suspeitas", [])
    if suspeitas:
        for s in suspeitas:
            print(f"ID: {s['id']} | Cliente: {s['cliente_id']} | Data: {s['data']} | Valor: {formatar_brl(s['valor'])}")
    else:
        print("Nenhuma transação suspeita encontrada.")
    print("=" * 78)


#Definindo função para salvar arquivo json

In [41]:
def salvar_json(resultado, nome_arquivo):
    """Salva o relatório estruturado em um arquivo JSON."""
    dados_json = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "periodo_analisado": resultado.get("periodo"),
        "resumo_mensal": resultado.get("resumo_mensal"),
        "transacoes_suspeitas": resultado.get("transacoes_suspeitas")
    }

    with open(nome_arquivo, "w", encoding="utf-8") as f:
        json.dump(dados_json, f, ensure_ascii=False, indent=4)
    print(f"\n[Sucesso] Relatório exportado com sucesso para '{nome_arquivo}'.")


#Executando o programa

In [42]:
# --- Célula de Execução Principal ---
if __name__ == "__main__":
    print("Iniciando processamento de transações da ClearBank...\n")
    transacoes_validas = ler_transacoes(ARQUIVO_CSV)

    if transacoes_validas:
        relatorio = gerar_relatorio(transacoes_validas)
        exibir_relatorio(relatorio)
        salvar_json(relatorio, ARQUIVO_JSON)

Iniciando processamento de transações da ClearBank...

Total de linhas lidas: 25
Linhas válidas: 18
Linhas inválidas: 7

============================== RELATÓRIO MENSAL ==============================
Período Analisado: 2026-01-05 até 2026-03-30 (84 dias entre a primeira e última transação)

Mês: 2026-01
  Transações: 5
  Total crédito: R$ 7.700,00
  Total débito:  R$ 1.430,50
  Saldo:         R$ 6.269,50
  Média:         R$ 1.826,10
  Maior valor:   R$ 4.200,00
  Menor valor:   R$ 50,00

Mês: 2026-02
  Transações: 6
  Total crédito: R$ 30.500,00
  Total débito:  R$ 1.070,00
  Saldo:         R$ 29.430,00
  Média:         R$ 5.261,67
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 300,00

Mês: 2026-03
  Transações: 7
  Total crédito: R$ 11.300,00
  Total débito:  R$ 1.109,90
  Saldo:         R$ 10.190,10
  Média:         R$ 1.772,84
  Maior valor:   R$ 5.000,00
  Menor valor:   R$ 60,00

============================ TRANSAÇÕES SUSPEITAS ============================
ID: 5 | Cliente: CLI